# 108 — Scaffold-Aware Delta Template Selection (Leave-One-Scaffold-Out)

**Motivation:** Standard delta-ML (nb97) selects templates purely by Tanimoto similarity. But if the highest-similarity neighbor shares the same Murcko scaffold, the delta-ML may be learning scaffold-specific memory rather than true generalization. Cross-scaffold delta — jumping between different scaffolds — may be more robust.

**Strategy:**
1. Same multi-template delta framework as nb97 (sim window [0.35, 0.90], K=10)
2. PRIMARY: prefer templates with a DIFFERENT Murcko scaffold than the query
3. FALLBACK: if fewer than MIN_CROSS_SCAFFOLD templates available, supplement with same-scaffold templates
4. MIN_CROSS_SCAFFOLD=2: need at least 2 cross-scaffold templates before falling back

**Hypothesis:** Cross-scaffold delta transfers generalize better to the test set (which is an analog expansion) than within-scaffold delta, because within-scaffold pairs may be overfitting to scaffold-specific activity patterns.

In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"): sys.stdout.reconfigure(encoding="utf-8")
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch
from pxr.paths import DATA_PROCESSED, SUBMISSIONS
SEED = 42; N_FOLDS = 5
LGBM = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)

In [2]:
def full_metrics(y_true, y_pred, cp=None, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr),
             Spearman=float(sp), Kendall=float(kt))
    if cp is not None and hasattr(cp, "iterrows") and len(cp) > 0:
        c=t=0
        for _,row in cp.iterrows():
            ia,ii = int(row.get("idx_active",-1)), int(row.get("idx_inactive",-1))
            if 0<=ia<len(yp) and 0<=ii<len(yp): c+=int(yp[ia]>yp[ii]); t+=1
        m["Cliff_acc"] = c/t if t else float("nan")
    if label:
        ca = f"  Cliff={m.get('Cliff_acc',float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R2={r2:.4f} "
              f"r={pr:.4f} rho={sp:.4f} tau={kt:.4f}{ca}")
    return m

In [3]:
from pxr.chem import compute_physchem
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)
X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
fps_tr = morgan_fp_batch(tr["smiles"].tolist()).astype(np.float32)
fps_te = morgan_fp_batch(te["smiles"].tolist()).astype(np.float32)
props = ["mw","logp","tpsa","hbd","hba","rotbonds","rings"]
print("Computing physchem...", flush=True)
phys_tr = tr["smiles"].map(compute_physchem).tolist()
phys_arr = np.array([[p.get(k,0) or 0 for k in props] for p in phys_tr], dtype=np.float32)

# Scaffold arrays
scaffold_arr = np.array(scaffolds)  # shape (N_train,)
te_scaffolds = te["smiles"].map(bemis_murcko).tolist()
te_scaffold_arr = np.array(te_scaffolds)

cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists() else pd.DataFrame())
if len(cliff_pairs) > 0:
    s2i = {s:i for i,s in enumerate(tr["smiles"].tolist())}
    ac = "cliff_active_smiles" if "cliff_active_smiles" in cliff_pairs.columns else "smiles_a"
    ic = "cliff_inactive_smiles" if "cliff_inactive_smiles" in cliff_pairs.columns else "smiles_b"
    cliff_pairs["idx_active"]   = cliff_pairs[ac].map(s2i)
    cliff_pairs["idx_inactive"] = cliff_pairs[ic].map(s2i)
    cliff_pairs = cliff_pairs.dropna(subset=["idx_active","idx_inactive"])
    cliff_pairs[["idx_active","idx_inactive"]] = cliff_pairs[["idx_active","idx_inactive"]].astype(int)
print(f"Train {len(tr):,}  Test {len(te):,}  Cliffs {len(cliff_pairs)}")
n_scaffolds = len(set(scaffolds))
print(f"Unique scaffolds in train: {n_scaffolds}  Test: {len(set(te_scaffolds))}")

Computing physchem...


Train 4,139  Test 513  Cliffs 0
Unique scaffolds in train: 3676  Test: 370


In [4]:
# --- Compute pairwise Tanimoto ---
SIM_LO = 0.35; SIM_HI = 0.90; MAX_PAIRS = 400_000
print("Computing pairwise Tanimoto (train x train)...", flush=True)
dot_tt = (fps_tr @ fps_tr.T).astype(np.float32)
rowsum = fps_tr.sum(1).astype(np.float32)
union_tt = rowsum[:,None] + rowsum[None,:] - dot_tt
tanimoto_tr = np.where(union_tt>0, dot_tt/union_tt, 0.0)
np.fill_diagonal(tanimoto_tr, 0.0)
i_idx, j_idx = np.where((tanimoto_tr >= SIM_LO) & (tanimoto_tr <= SIM_HI))
mask_upper = i_idx < j_idx
i_idx, j_idx = i_idx[mask_upper], j_idx[mask_upper]
print(f"Pairs in sim window [{SIM_LO},{SIM_HI}]: {len(i_idx):,}")

# Cross-scaffold vs within-scaffold pair statistics
is_cross = scaffold_arr[i_idx] != scaffold_arr[j_idx]
print(f"  Cross-scaffold pairs: {is_cross.sum():,} ({100*is_cross.mean():.1f}%)")
print(f"  Within-scaffold pairs: {(~is_cross).sum():,} ({100*(~is_cross).mean():.1f}%)")

rng = np.random.default_rng(SEED)
if len(i_idx) > MAX_PAIRS:
    sel = rng.choice(len(i_idx), MAX_PAIRS, replace=False)
    i_idx, j_idx = i_idx[sel], j_idx[sel]

Computing pairwise Tanimoto (train x train)...


Pairs in sim window [0.35,0.9]: 5,177
  Cross-scaffold pairs: 4,533 (87.6%)
  Within-scaffold pairs: 644 (12.4%)


In [5]:
# --- Feature helpers (same as nb97/nb104) ---
def compress_fp(fp, out_dim=64):
    N, D = fp.shape; block = D // out_dim
    return fp[:, :block*out_dim].reshape(N, out_dim, block).mean(-1).astype(np.float32)

def make_delta_feats(fp_anchor, fp_query, sim_col, anchor_pec50, phys_diff):
    fp_common = np.minimum(fp_anchor, fp_query).astype(np.float32)
    fp_diff   = np.abs(fp_anchor - fp_query).astype(np.float32)
    c64 = compress_fp(fp_common)
    d64 = compress_fp(fp_diff)
    return np.hstack([c64, d64, sim_col, anchor_pec50[:,None], phys_diff])

sims_ij = tanimoto_tr[i_idx, j_idx][:,None]
phys_diff_ij = phys_arr[j_idx] - phys_arr[i_idx]
F_ij = make_delta_feats(fps_tr[i_idx], fps_tr[j_idx], sims_ij, y_tr[i_idx], phys_diff_ij)
F_ji = make_delta_feats(fps_tr[j_idx], fps_tr[i_idx], sims_ij, y_tr[j_idx], -phys_diff_ij)
F_all = np.vstack([F_ij, F_ji])
y_all = np.concatenate([y_tr[j_idx]-y_tr[i_idx], y_tr[i_idx]-y_tr[j_idx]])
print(f"Delta dataset: {F_all.shape}  delta range [{y_all.min():.2f}, {y_all.max():.2f}]")

print("Training global delta LGBM...", flush=True)
DELTA_LGBM = dict(n_estimators=600, num_leaves=63, learning_rate=0.05,
                  min_child_samples=20, subsample=0.8, colsample_bytree=0.7,
                  reg_alpha=0.05, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)
delta_model = lgb.LGBMRegressor(**DELTA_LGBM)
delta_model.fit(F_all, y_all, callbacks=[lgb.log_evaluation(-1)])
print("Delta model trained.", flush=True)

Delta dataset: (10354, 137)  delta range [-4.68, 4.68]
Training global delta LGBM...


Delta model trained.


In [6]:
# --- Scaffold-aware multi-template prediction ---
K_NEIGHBORS = 10
MIN_CROSS_SCAFFOLD = 2  # minimum cross-scaffold templates before falling back to same-scaffold

def scaffold_aware_delta_predict(fps_query, fps_ref, y_ref, phys_query, phys_ref,
                                  sim_matrix, scaffolds_query, scaffolds_ref,
                                  delta_model, direct_preds,
                                  sim_lo=SIM_LO, sim_hi=SIM_HI,
                                  k=K_NEIGHBORS, min_cross=MIN_CROSS_SCAFFOLD):
    """
    PRIMARY: use cross-scaffold templates (different Murcko scaffold than query)
    FALLBACK: if < min_cross cross-scaffold templates, include same-scaffold templates
    Returns: (preds, n_cross, n_total, mode) where mode: 0=fallback, 1=direct, 2=cross_only
    """
    N = len(fps_query)
    preds = np.full(N, np.nan)
    n_cross    = np.zeros(N, dtype=int)
    n_total    = np.zeros(N, dtype=int)
    mode       = np.zeros(N, dtype=int)  # 0=fallback-mixed, 1=no-template, 2=cross-only

    for qi in range(N):
        sim_row = sim_matrix[qi]
        q_scaffold = scaffolds_query[qi]

        # Find all candidates in sim window
        cand_mask = (sim_row >= sim_lo) & (sim_row <= sim_hi)
        cand_idx_all = np.where(cand_mask)[0]
        if len(cand_idx_all) == 0:
            preds[qi] = direct_preds[qi]
            mode[qi] = 1
            continue

        # Separate cross- and same-scaffold
        cross_mask = np.array([scaffolds_ref[ci] != q_scaffold for ci in cand_idx_all])
        cross_idx  = cand_idx_all[cross_mask]
        same_idx   = cand_idx_all[~cross_mask]

        if len(cross_idx) > 0 and len(cross_idx) >= min_cross:
            # Use only cross-scaffold templates (take top-k by similarity)
            top_k = np.argsort(-sim_row[cross_idx])[:k]
            sel_idx = cross_idx[top_k]
            mode[qi] = 2
        else:
            # Fallback: combine cross + same-scaffold, rank by similarity
            cand_sims_all = sim_row[cand_idx_all]
            top_k = np.argsort(-cand_sims_all)[:k]
            sel_idx = cand_idx_all[top_k]
            mode[qi] = 0

        cand_sims = sim_row[sel_idx]
        n_total[qi] = len(sel_idx)
        n_cross[qi] = int(np.sum([scaffolds_ref[ci] != q_scaffold for ci in sel_idx]))

        fp_q_rep = np.tile(fps_query[qi:qi+1], (len(sel_idx), 1))
        fp_refs  = fps_ref[sel_idx]
        sims_col = cand_sims[:,None]
        anc_pec50 = y_ref[sel_idx]
        phys_d = phys_query[qi:qi+1] - phys_ref[sel_idx]
        F_k = make_delta_feats(fp_refs, fp_q_rep, sims_col, anc_pec50, phys_d)
        delta_k = delta_model.predict(F_k)
        template_preds = y_ref[sel_idx] + delta_k
        weights = cand_sims ** 2
        preds[qi] = np.average(template_preds, weights=weights)

    return preds, n_cross, n_total, mode

print(f"Scaffold-aware delta function ready (K={K_NEIGHBORS}, min_cross={MIN_CROSS_SCAFFOLD})")

Scaffold-aware delta function ready (K=10, min_cross=2)


In [7]:
# --- Scaffold 5-fold CV ---
print("\n=== Scaffold 5-fold CV ===", flush=True)
oof_loso   = np.full(len(y_tr), np.nan)
oof_direct = np.full(len(y_tr), np.nan)
oof_mode   = np.full(len(y_tr), -1, dtype=int)
oof_n_cross = np.zeros(len(y_tr), dtype=int)

for fold, (tr_idx, va_idx) in enumerate(splits):
    m_dir = lgb.train(LGBM, lgb.Dataset(X_tr[tr_idx], label=y_tr[tr_idx]),
                      valid_sets=[lgb.Dataset(X_tr[va_idx], label=y_tr[va_idx])],
                      callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
    oof_direct[va_idx] = m_dir.predict(X_tr[va_idx])

    fps_va = fps_tr[va_idx]; fps_ft = fps_tr[tr_idx]
    dot_vf = (fps_va @ fps_ft.T).astype(np.float32)
    rs_v = fps_va.sum(1)[:,None]; rs_f = fps_ft.sum(1)[None,:]
    sim_vf = dot_vf / np.maximum(rs_v + rs_f - dot_vf, 1e-6)

    scaffolds_va = [scaffold_arr[i] for i in va_idx]
    scaffolds_ft = [scaffold_arr[i] for i in tr_idx]

    preds_loso, nc, nt, md = scaffold_aware_delta_predict(
        fps_va, fps_ft, y_tr[tr_idx], phys_arr[va_idx], phys_arr[tr_idx],
        sim_vf, scaffolds_va, scaffolds_ft,
        delta_model, oof_direct[va_idx]
    )
    oof_loso[va_idx]    = preds_loso
    oof_mode[va_idx]    = md
    oof_n_cross[va_idx] = nc

    r_dir  = rae(y_tr[va_idx], oof_direct[va_idx])
    r_loso = rae(y_tr[va_idx], oof_loso[va_idx])
    n_cross_only = int((md==2).sum())
    n_mixed = int((md==0).sum())
    n_direct = int((md==1).sum())
    print(f"  fold {fold+1}  direct={r_dir:.4f}  loso={r_loso:.4f}  "
          f"cross_only={n_cross_only}/mixed={n_mixed}/direct={n_direct}  "
          f"avg_cross={float(nc.mean()):.1f}",
          flush=True)

print(f"\nOverall mode breakdown: cross_only={int((oof_mode==2).sum())} "
      f"mixed={int((oof_mode==0).sum())} direct={int((oof_mode==1).sum())}")
m_dir  = full_metrics(y_tr, oof_direct, cliff_pairs, "direct_lgbm")
m_loso = full_metrics(y_tr, oof_loso,   cliff_pairs, "scaffold_loso_delta")


=== Scaffold 5-fold CV ===


  fold 1  direct=0.4982  loso=0.2912  cross_only=341/mixed=192/direct=295  avg_cross=1.9


  fold 2  direct=0.5759  loso=0.3193  cross_only=320/mixed=197/direct=311  avg_cross=1.8


  fold 3  direct=0.6021  loso=0.3612  cross_only=312/mixed=180/direct=336  avg_cross=1.7


  fold 4  direct=0.5665  loso=0.3294  cross_only=304/mixed=205/direct=319  avg_cross=1.7


  fold 5  direct=0.6033  loso=0.3461  cross_only=321/mixed=183/direct=323  avg_cross=1.7



Overall mode breakdown: cross_only=1598 mixed=957 direct=1584
  [direct_lgbm] RAE=0.5643 MAE=0.5134 R2=0.5991 r=0.7740 rho=0.7268 tau=0.5345
  [scaffold_loso_delta] RAE=0.3266 MAE=0.2972 R2=0.8178 r=0.9060 rho=0.8754 tau=0.7237


In [8]:
# --- Sweep MIN_CROSS_SCAFFOLD threshold ---
print("\nSweeping min_cross_scaffold threshold...", flush=True)
best_min_cross, best_rae_v = MIN_CROSS_SCAFFOLD, full_metrics(y_tr, oof_loso)["RAE"]
for mc in [0, 1, 2, 3, 5, 8]:
    oof_mc = np.full(len(y_tr), np.nan)
    for fold, (tr_idx, va_idx) in enumerate(splits):
        fps_va = fps_tr[va_idx]; fps_ft = fps_tr[tr_idx]
        dot_vf = (fps_va @ fps_ft.T).astype(np.float32)
        rs_v = fps_va.sum(1)[:,None]; rs_f = fps_ft.sum(1)[None,:]
        sim_vf = dot_vf / np.maximum(rs_v + rs_f - dot_vf, 1e-6)
        scaffolds_va = [scaffold_arr[i] for i in va_idx]
        scaffolds_ft = [scaffold_arr[i] for i in tr_idx]
        p_mc, _, _, _ = scaffold_aware_delta_predict(
            fps_va, fps_ft, y_tr[tr_idx], phys_arr[va_idx], phys_arr[tr_idx],
            sim_vf, scaffolds_va, scaffolds_ft,
            delta_model, oof_direct[va_idx], min_cross=mc
        )
        oof_mc[va_idx] = p_mc
    mask = np.isfinite(oof_mc)
    r = rae(y_tr[mask], oof_mc[mask])
    print(f"  min_cross={mc}  RAE={r:.4f}")
    if r < best_rae_v:
        best_rae_v, best_min_cross = r, mc

print(f"\nBest min_cross={best_min_cross}  OOF RAE={best_rae_v:.4f}")

# Recompute OOF with best min_cross
if best_min_cross != MIN_CROSS_SCAFFOLD:
    oof_loso_best = np.full(len(y_tr), np.nan)
    for fold, (tr_idx, va_idx) in enumerate(splits):
        fps_va = fps_tr[va_idx]; fps_ft = fps_tr[tr_idx]
        dot_vf = (fps_va @ fps_ft.T).astype(np.float32)
        rs_v = fps_va.sum(1)[:,None]; rs_f = fps_ft.sum(1)[None,:]
        sim_vf = dot_vf / np.maximum(rs_v + rs_f - dot_vf, 1e-6)
        scaffolds_va = [scaffold_arr[i] for i in va_idx]
        scaffolds_ft = [scaffold_arr[i] for i in tr_idx]
        p_best, _, _, _ = scaffold_aware_delta_predict(
            fps_va, fps_ft, y_tr[tr_idx], phys_arr[va_idx], phys_arr[tr_idx],
            sim_vf, scaffolds_va, scaffolds_ft,
            delta_model, oof_direct[va_idx], min_cross=best_min_cross
        )
        oof_loso_best[va_idx] = p_best
    oof_loso = oof_loso_best

# --- Blend sweep ---
best_alpha, best_rae_bl = 0.0, full_metrics(y_tr, oof_direct)["RAE"]
for alpha in np.arange(0.0, 1.05, 0.1):
    blended = alpha*oof_loso + (1-alpha)*oof_direct
    mask = np.isfinite(blended)
    r = rae(y_tr[mask], blended[mask])
    print(f"  alpha={alpha:.1f}  RAE={r:.4f}")
    if r < best_rae_bl:
        best_rae_bl, best_alpha = r, alpha

oof = best_alpha*oof_loso + (1-best_alpha)*oof_direct
m_blend = full_metrics(y_tr, oof, cliff_pairs, f"blend(a={best_alpha:.1f})")
print(f"\nBest blend alpha={best_alpha:.1f}  OOF RAE={best_rae_bl:.4f}")
print("\n" + pd.DataFrame([m_dir, m_loso, m_blend],
                           index=["direct",f"loso_mc{best_min_cross}",f"blend_{best_alpha:.1f}"]).round(4).to_string())


Sweeping min_cross_scaffold threshold...


  min_cross=0  RAE=0.3266


  min_cross=1  RAE=0.3266


  min_cross=2  RAE=0.3266


  min_cross=3  RAE=0.3266


  min_cross=5  RAE=0.3266


  min_cross=8  RAE=0.3266

Best min_cross=0  OOF RAE=0.3266


  alpha=0.0  RAE=0.5643
  alpha=0.1  RAE=0.5381
  alpha=0.2  RAE=0.5120
  alpha=0.3  RAE=0.4863
  alpha=0.4  RAE=0.4608
  alpha=0.5  RAE=0.4357
  alpha=0.6  RAE=0.4113
  alpha=0.7  RAE=0.3878
  alpha=0.8  RAE=0.3653
  alpha=0.9  RAE=0.3446
  alpha=1.0  RAE=0.3266
  [blend(a=1.0)] RAE=0.3266 MAE=0.2972 R2=0.8178 r=0.9060 rho=0.8754 tau=0.7237

Best blend alpha=1.0  OOF RAE=0.3266

              RAE     MAE      R2  Pearson  Spearman  Kendall
direct     0.5643  0.5134  0.5991    0.774    0.7268   0.5345
loso_mc0   0.3266  0.2972  0.8178    0.906    0.8754   0.7237
blend_1.0  0.3266  0.2972  0.8178    0.906    0.8754   0.7237


In [9]:
# --- Final test predictions ---
print("\nFitting final direct LGBM on all train...", flush=True)
m_final = lgb.train(LGBM, lgb.Dataset(X_tr, label=y_tr), callbacks=[lgb.log_evaluation(-1)])
te_direct = m_final.predict(X_te)

dot_tet = (fps_te @ fps_tr.T).astype(np.float32)
rs_te = fps_te.sum(1)[:,None]; rs_tr_v = fps_tr.sum(1)[None,:]
sim_te_tr = dot_tet / np.maximum(rs_te + rs_tr_v - dot_tet, 1e-6)
phys_te = np.array([[p.get(k,0) or 0 for k in props]
                     for p in te["smiles"].map(compute_physchem)], dtype=np.float32)

print("Running scaffold-aware delta on test...", flush=True)
te_loso, te_nc, te_nt, te_mode = scaffold_aware_delta_predict(
    fps_te, fps_tr, y_tr, phys_te, phys_arr,
    sim_te_tr, te_scaffolds, scaffolds,
    delta_model, te_direct, min_cross=best_min_cross
)
print(f"Test: cross_only={int((te_mode==2).sum())} mixed={int((te_mode==0).sum())} "
      f"direct={int((te_mode==1).sum())}  avg_cross={float(te_nc.mean()):.1f}")

te_preds = best_alpha*te_loso + (1-best_alpha)*te_direct
te_preds = np.clip(te_preds, y_tr.min()-0.5, y_tr.max()+0.5)

np.save(DATA_PROCESSED/"oof_delta_loso.npy", oof)
np.save(DATA_PROCESSED/"te_oof_delta_loso.npy", te_preds)
sub = pd.DataFrame({"Molecule Name": te["name"].values, "pEC50": te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"108_delta_leave_one_scaffold_out.csv"; sub.to_csv(p, index=False)
print(f"Saved {p}")
print(f"Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")
print(f"\n*** nb108 OOF RAE = {m_blend['RAE']:.4f} ***")


Fitting final direct LGBM on all train...


Running scaffold-aware delta on test...


Test: cross_only=493 mixed=17 direct=3  avg_cross=3.0
Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\108_delta_leave_one_scaffold_out.csv
Test: min=2.80 med=4.72 max=5.74

*** nb108 OOF RAE = 0.3266 ***
